In [ ]:
from src.retrievers.tf_idf import TFIDF
from src.retrievers.base_retriever import Retriever

docs = [{'text': ['the cat sat on the mat',
                  'the dog sat on the log',
                  'cats and dogs'],
         'document_id': ['d0', 'd1', 'd2']}]

class DevTFIDF(TFIDF):
    def __init__(self, documents):
        Retriever.__init__(self, documents)   # only builds self.index, skips the TFIDF stubs
        
        self.corpus_vocabulary = self.GetCorpusVocabulary()
        self.idf = self.GetInverseDocumentFrequencies()

        # Only run if we are in TFIDF class
        # if type(self) == TFIDF:
        self.tfidf_matrix = self.GetDocumentsTFIDFVectors()

t = DevTFIDF(docs) # instantiate the class, to access the methods

In [2]:
print(t.PreprocessText("The cat sat on the mat."))
print(t.GetQueryVocabulary("cat on the mat"))
print(t.GetCorpusVocabulary()) 


['the', 'cat', 'sat', 'on', 'the', 'mat.']
['on', 'mat', 'the', 'cat']
{'cats', 'mat', 'dog', 'cat', 'sat', 'and', 'the', 'on', 'dogs', 'log'}


In [3]:
from src.data_utils.document import Document

doc = Document(text="the cat sat on the mat", _id="d0")

print(t.GetDocumentVocabulary(doc))
print(t.GetDocumentTermCounts(doc))


{'mat', 'sat', 'cat', 'the', 'on'}
{'mat': 1, 'sat': 1, 'cat': 1, 'the': 2, 'on': 1}


In [4]:
print(t.GetDocumentFrequencies())
print(t.GetInverseDocumentFrequencies())

{'mat': 1, 'sat': 2, 'cat': 1, 'the': 2, 'on': 2, 'dogs': 2, 'dog': 1, 'log': 1, 'cats': 1, 'and': 1}
{'mat': np.float64(1.0986122886681098), 'sat': np.float64(0.4054651081081644), 'cat': np.float64(1.0986122886681098), 'the': np.float64(0.4054651081081644), 'on': np.float64(0.4054651081081644), 'dogs': np.float64(0.4054651081081644), 'dog': np.float64(1.0986122886681098), 'log': np.float64(1.0986122886681098), 'cats': np.float64(1.0986122886681098), 'and': np.float64(1.0986122886681098)}


### Queries

In [5]:
from src.data_utils.query import Query


class DevQuery(Query):
    def __init__(self, text: str, id: str, relevant_document_ids: list[str], label: str):
        
        self.text = text
        self.id = id
        self.relevant_document_ids = relevant_document_ids
        self.label = label


queries = [
    DevQuery(text='unrelated dogs',         id='q0', relevant_document_ids=['d0'], label='q0'),
    DevQuery(text='hello dogs',            id='q1', relevant_document_ids=['d1'], label='q1'),
    DevQuery(text='cats and dogs',          id='q2', relevant_document_ids=['d2'], label='q2'),
]

q0,q1,q2 = queries

In [6]:
q1.getQuery()

'hello dogs'

In [14]:
[t.GetQueryVocabulary(q.getQuery()) for q in queries]


[['dogs', 'unrelated'], ['dogs', 'hello'], ['cats', 'dogs', 'and']]

In [9]:
[(q.getQuery()) for q in queries]

['unrelated dogs', 'hello dogs', 'cats and dogs']

In [10]:
docs

[{'text': ['the cat sat on the mat dogs dogs dogs',
   'the dog sat on the log',
   'cats and dogs'],
  'document_id': ['d0', 'd1', 'd2']}]

In [11]:
[t.QueryToVector([q.getQuery()]) for q in queries]


[<Compressed Sparse Row sparse matrix of dtype 'float32'
 	with 1 stored elements and shape (1, 10)>,
 <Compressed Sparse Row sparse matrix of dtype 'float32'
 	with 1 stored elements and shape (1, 10)>,
 <Compressed Sparse Row sparse matrix of dtype 'float32'
 	with 3 stored elements and shape (1, 10)>]

In [12]:
q1.getQuery()

'hello dogs'

In [13]:
[t.CalculateScores([q.getQuery()]) for q in queries]

[array([[0.4054651, 0.       , 0.4054651]], dtype=float32),
 array([[0.4054651, 0.       , 0.4054651]], dtype=float32),
 array([[0.4054651, 0.       , 2.6026897]], dtype=float32)]